In [1]:
from transformers import (
    TapasTokenizer, TapasForQuestionAnswering,
)
from datasets import load_dataset
import torch
import pandas as pd
from tqdm import tqdm

d:\Sorbonne\M2-MIND\MEDS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/wikitablequestions", trust_remote_code=True)
test_data = dataset["test"]

print("Number of test samples:", len(test_data))
print("Example:", test_data[0])


Number of test samples: 4344
Example: {'id': 'nu-0', 'question': 'which country had the most cyclists finish within the top 10?', 'answers': ['Italy'], 'table': {'header': ['Rank', 'Cyclist', 'Team', 'Time', 'UCI ProTour\\nPoints'], 'rows': [['1', 'Alejandro Valverde\xa0(ESP)', "Caisse d'Epargne", '5h 29\' 10"', '40'], ['2', 'Alexandr Kolobnev\xa0(RUS)', 'Team CSC Saxo Bank', 's.t.', '30'], ['3', 'Davide Rebellin\xa0(ITA)', 'Gerolsteiner', 's.t.', '25'], ['4', 'Paolo Bettini\xa0(ITA)', 'Quick Step', 's.t.', '20'], ['5', 'Franco Pellizotti\xa0(ITA)', 'Liquigas', 's.t.', '15'], ['6', 'Denis Menchov\xa0(RUS)', 'Rabobank', 's.t.', '11'], ['7', 'Samuel Sánchez\xa0(ESP)', 'Euskaltel-Euskadi', 's.t.', '7'], ['8', 'Stéphane Goubert\xa0(FRA)', 'Ag2r-La Mondiale', '+ 2"', '5'], ['9', 'Haimar Zubeldia\xa0(ESP)', 'Euskaltel-Euskadi', '+ 2"', '3'], ['10', 'David Moncoutié\xa0(FRA)', 'Cofidis', '+ 2"', '1']], 'name': 'csv/203-csv/733.tsv'}}


In [4]:
tapas_tokenizer = TapasTokenizer.from_pretrained("google/tapas-large-finetuned-wtq")
tapas_model = TapasForQuestionAnswering.from_pretrained("google/tapas-large-finetuned-wtq")


The code below is a basic test with like 30% accuracy but the paper says it can reach 48%. And we used the wtq finetuned for this test so the problem is most likely coming from us (except if the base model doesnt perform as well i need to double check that). Anyways maybe a better config could do the trick, see here: https://huggingface.co/docs/transformers/model_doc/tapas

In [5]:
def normalize_answer(a):# normalize answers to try and avoid errors because of mismatching
    if a is None:
        return "" # default
    a = str(a).strip().lower() 
    # remove commas
    a = a.replace(",", "")
    # handle percentages
    if a.endswith("%"):
        try:
            return float(a[:-1]) / 100 # turn into a normal float
        except:
            return a
    # convert numeric strings to float
    try:
        return float(a)
    except:
        return a

In [6]:
def compute_final_answer(table, answer_coords, aggregation):
    """
    Convert selected cell coordinates + aggregation into final predicted answer.
    """
    if not answer_coords:
        return None
    
    values = [table.iat[row, col] for row, col in answer_coords]
    
    if aggregation == "NONE":
        return " ".join(map(str, values))
    elif aggregation == "COUNT":
        return len(values)
    elif aggregation == "SUM":
        try:
            return sum(float(v) for v in values)
        except:
            return " ".join(map(str, values))
    elif aggregation == "AVERAGE":
        try:
            return sum(float(v) for v in values) / len(values)
        except:
            return " ".join(map(str, values))
    else:
        return " ".join(map(str, values))

In [7]:
def test_model(model, tokenizer, data):
    correct = 0
    total = len(data)
    total_skipped = 0
    print("Evaluating on", total, "samples...")

    progress = tqdm(data, desc="Evaluating", ncols=100)

    for ex in progress:
        headers = ex["table"]["header"]
        rows = ex["table"]["rows"]

        # handle irregular row lengths
        max_len = len(headers)
        clean_rows = [r + [""] * (max_len - len(r)) for r in rows]
        table = pd.DataFrame(clean_rows, columns=headers)
        question = ex["question"]

        # Tokenize 
        inputs = tokenizer(
            table=table,
            queries=[question],
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        # Run model
        try:
            with torch.no_grad():
                outputs = model(**inputs)
        except IndexError:
            total_skipped += 1
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue
        logits = outputs.logits.cpu()

        predictions = tokenizer.convert_logits_to_predictions(inputs, logits)
        if len(predictions) == 2:
            predicted_answer_coordinates, predicted_aggregation = predictions
        else:
            predicted_answer_coordinates = predictions[0]
            predicted_aggregation = ["NONE"]

        # Compute final answer using aggregation
        predicted_answer = compute_final_answer(table, predicted_answer_coordinates[0], predicted_aggregation[0])

        # Normalize predicted and gold answers
        normalized_pred = normalize_answer(predicted_answer)
        normalized_gold = set(map(normalize_answer, ex["answers"]))

        if normalized_pred in normalized_gold:
            correct += 1

        progress.set_postfix({
            "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
            "skipped": total_skipped
        })

    return correct, total, total_skipped

In [8]:
dev_test = dataset["validation"] # this is what they use in the paper for the accuracy

print("Number of test samples:", len(dev_test))

Number of test samples: 2831


In [23]:
correct, total, total_skipped = test_model(tapas_model, tapas_tokenizer, dev_test)
print(f"Accuracy: {correct / (total - total_skipped) * 100:.2f}%, out of {total - total_skipped} samples (skipped {total_skipped})")

Evaluating on 2831 samples...


Evaluating: 100%|███████████████████| 2831/2831 [32:21<00:00,  1.46it/s, accuracy=32.44%, skipped=6]

Accuracy: 32.42%, out of 2825 samples (skipped 6)


So the accuracy is lower than what we see on the paper, on the test set they get 42% accuracy so we are way under. Most probbaly because of the way we handle answers? But we normalized and filtered so idk why it would be worse. Also i think part of the issue could be because of the amount of tokens we pass to the model. TAPAS can only handle like 512 tokens at a time and some tables are larger so its possible it completely gets it wrong there. Also we added that fnuction in case it returns an operation to do on the table but i dont think its being used very much. Anyways idk how they did it in the paper because its not mentionned explicitely. On the bright side if we get similar results (with regards to the results in papers) for TAPEX and TURL then we can still analyze results. But if TAPEX works as intended then it is indeed a TAPAS problem.

Paper is here for TAPAS: https://arxiv.org/pdf/2004.02349

Anyways lets test TAPEX now and see how it performs.

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("microsoft/tapex-large-finetuned-wtq")
model = AutoModelForSeq2SeqLM.from_pretrained("microsoft/tapex-large-finetuned-wtq")

In [10]:
def test_tapex_model(model, tokenizer, data, device="cuda" if torch.cuda.is_available() else "cpu"):
    model.to(device)
    model.eval()

    correct = 0
    total = len(data)
    total_skipped = 0
    print("Evaluating on", total, "samples...")

    progress = tqdm(data, desc="Evaluating", ncols=100)

    for ex in progress:
        headers = ex["table"]["header"]
        rows = ex["table"]["rows"]

        # Handle irregular row lengths
        max_len = len(headers)
        clean_rows = [r + [""] * (max_len - len(r)) for r in rows]
        table = pd.DataFrame(clean_rows, columns=headers)
        question = ex["question"]

        # Tokenize
        inputs = tokenizer(
            table,
            question,
            return_tensors="pt",
            truncation=True,
            padding=True,
        ).to(device)

        # Generate answer
        try:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=50)
            pred_answer_raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
        except Exception as e:
            total_skipped += 1
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
                "skipped": total_skipped,
                "error": str(e)
            })
            continue

        # Split predicted answer into multiple answers if comma/semicolon-separated
        pred_answers = [normalize_answer(a) for a in pred_answer_raw.replace(';', ',').split(',') if a.strip()]

        # Normalize gold answers
        normalized_gold = set(map(normalize_answer, ex["answers"]))

        # Count as correct if **any** predicted answer matches a gold answer
        if any(a in normalized_gold for a in pred_answers):
            correct += 1

        progress.set_postfix({
            "accuracy": f"{(correct / max(1, (progress.n - total_skipped))) * 100:.2f}%",
            "skipped": total_skipped
        })

    return correct, total, total_skipped

In [11]:
dev_test = dataset["validation"]

In [87]:
test_tapex_model(model, tokenizer, dev_test, device)

Evaluating on 2831 samples...


Evaluating: 100%|███████████████████| 2831/2831 [05:38<00:00,  8.36it/s, accuracy=56.17%, skipped=0]


(1589, 2831, 0)

Aaay as we can see we got 56% accuracy (the paper says 57% but its an average so we are spot on).

Paper is here: https://arxiv.org/pdf/2107.07653

Very cool

In [33]:
# lets do a quick test same model but on train data bc we will be using train for cluster testing
# do it only on the first 1000 samples to save time
# its only 10% but its just to see if the model is working properly on train data
test_tapex_model(model, tokenizer, dataset["train"].select(range(1000)), device)

Evaluating on 1000 samples...


Evaluating: 100%|███████████████████| 1000/1000 [01:58<00:00,  8.41it/s, accuracy=86.69%, skipped=0]


(866, 1000, 0)

Stopped the train early but yeah we get like 86% accuracy.

OK now let's try and get scores **relative to clusters**. This time we are going to start with tapex since it works as intended.

In [34]:
# recap of what the data looks like
# we are using the train data now bc thats the only data we have clusters on
print(dataset["train"][0]["question"])
print(dataset["train"][0]["table"]["header"])
print(dataset["train"][0]["table"]["rows"])
print(dataset["train"][0]["answers"])

what was the last year where this team was a part of the usl a-league?
['Year', 'Division', 'League', 'Regular Season', 'Playoffs', 'Open Cup', 'Avg. Attendance']
[['2001', '2', 'USL A-League', '4th, Western', 'Quarterfinals', 'Did not qualify', '7,169'], ['2002', '2', 'USL A-League', '2nd, Pacific', '1st Round', 'Did not qualify', '6,260'], ['2003', '2', 'USL A-League', '3rd, Pacific', 'Did not qualify', 'Did not qualify', '5,871'], ['2004', '2', 'USL A-League', '1st, Western', 'Quarterfinals', '4th Round', '5,628'], ['2005', '2', 'USL First Division', '5th', 'Quarterfinals', '4th Round', '6,028'], ['2006', '2', 'USL First Division', '11th', 'Did not qualify', '3rd Round', '5,575'], ['2007', '2', 'USL First Division', '2nd', 'Semifinals', '2nd Round', '6,851'], ['2008', '2', 'USL First Division', '11th', 'Did not qualify', '1st Round', '8,567'], ['2009', '2', 'USL First Division', '1st', 'Semifinals', '3rd Round', '9,734'], ['2010', '2', 'USSF D-2 Pro League', '3rd, USL (3rd)', 'Quart

We also have a new .tsv file which contains the same exact values as the ["questions"] variable but with an extra key: clusters

In [35]:
cluster_df = pd.read_csv("training_clusters.tsv", sep="\t")
# print the column names
print(cluster_df.columns)
print(cluster_df.iloc[0]) 

Index(['id', 'utterance', 'context', 'targetValue', 'cluster'], dtype='object')
id                                                          nt-0
utterance      what was the last year where this team was a p...
context                                      csv/204-csv/590.csv
targetValue                                                 2004
cluster                                                    Noise
Name: 0, dtype: object


So we can see compared to the previous cell that it contains the same question (same order too). 

Only issue is its not 100% reliable since this obvious sports themed question is labeled as noise. 
If we check in the dataset we still see many are labeled correctly, like the second one (see .tsv file for more).

Now lets adapt the tapex testing loop for this change.

In [12]:
import os

def test_clustered_tapex(model, tokenizer, cluster_tsv_path, wtq_root, device):
    model.to(device)
    model.eval()

    # Load cluster data
    cluster_df = pd.read_csv(cluster_tsv_path, sep="\t")

    correct = 0
    total = len(cluster_df)
    total_skipped = 0

    # Track per-cluster accuracy
    cluster_stats = {c: {"correct": 0, "total": 0} for c in cluster_df["cluster"].unique()}

    print(f"Evaluating on {total} samples...")

    progress = tqdm(cluster_df.itertuples(), total=total, desc="Evaluating", ncols=120)
    tracker = 0
    for row in progress:
        question = row.utterance
        answer = str(row.targetValue).strip()
        cluster = row.cluster
        csv_path = os.path.join(wtq_root, *row.context.split("/"))
        csv_path = os.path.normpath(csv_path)

        # Read table
        try:
            table = pd.read_csv(csv_path, sep=",", quotechar='"', on_bad_lines='skip')
        except Exception as e:
            total_skipped += 1
            print(f"Failed to read CSV: {csv_path}")
            print(f"Error: {e}")
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue

        # Tokenize and generate
        table = table.fillna("").astype(str)
        try:
            inputs = tokenizer(
                table,
                question,
                return_tensors="pt",
                truncation=True,
                padding=True,
            ).to(device)

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=50)

            pred_answer_raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
            pred_answers = [normalize_answer(a) for a in pred_answer_raw.replace(';', ',').split(',') if a.strip()]
            gold_answers = {normalize_answer(answer)}

            if any(a in gold_answers for a in pred_answers):
                correct += 1
                cluster_stats[cluster]["correct"] += 1
            cluster_stats[cluster]["total"] += 1

        except Exception as e:
            print(f"Error processing sample ID {row.id}: {e}")
            total_skipped += 1
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue

        progress.set_postfix({
            "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
            "skipped": total_skipped
        })
        tracker += 1

        """ if tracker > 300:
            break """

    global_acc = (correct / max(1, total - total_skipped)) * 100

    # Per-cluster accuracy
    print("\nPer-cluster accuracy:")
    for c, stats in cluster_stats.items():
        if stats["total"] > 0:
            acc = (stats["correct"] / stats["total"]) * 100
            print(f"{c:<15}: {acc:.2f}% ({stats['correct']}/{stats['total']})")

    return global_acc, cluster_stats, total_skipped

In [13]:
wtq_root = "WikiTableQuestions"
cluster_tsv_path = "training_clusters.tsv"

In [86]:
global_acc, cluster_stats, total_skipped = test_clustered_tapex(model, tokenizer, cluster_tsv_path, wtq_root, device)

Evaluating on 14149 samples...


Evaluating:  14%|█████▍                                | 2035/14149 [03:58<18:10, 11.11it/s, accuracy=74.74%, skipped=1]

Failed to read CSV: WikiTableQuestions\csv\204-csv\870.csv
Error: Error tokenizing data. C error: EOF inside string starting at row 17


Evaluating:  24%|█████████▏                            | 3405/14149 [06:35<16:16, 11.00it/s, accuracy=74.80%, skipped=2]

Failed to read CSV: WikiTableQuestions\csv\204-csv\870.csv
Error: Error tokenizing data. C error: EOF inside string starting at row 17


...

Failed to read CSV: WikiTableQuestions\csv\204-csv\870.csv
Error: Error tokenizing data. C error: EOF inside string starting at row 17


Evaluating:  74%|███████████████████████████▏         | 10416/14149 [20:13<07:28,  8.32it/s, accuracy=74.25%, skipped=9]

Failed to read CSV: WikiTableQuestions\csv\204-csv\870.csv
Error: Error tokenizing data. C error: EOF inside string starting at row 17


Evaluating: 100%|█████████████████████████████████████| 14149/14149 [27:27<00:00,  8.59it/s, accuracy=74.50%, skipped=9]


Per-cluster accuracy:
Noise          : 77.41% (2382/3077)
Sports         : 75.31% (4707/6250)
General knowledge: 70.64% (818/1158)
Politics       : 71.36% (999/1400)
Film           : 72.97% (594/814)
Music          : 71.03% (765/1077)
Science        : 68.82% (117/170)
Geography      : 77.84% (151/194)


In [14]:
import os

def new_clustered_tapex(model, tokenizer, cluster_tsv_path, wtq_root, device):
    model.to(device)
    model.eval()

    # Load cluster data
    cluster_df = pd.read_csv(cluster_tsv_path, sep=",")

    correct = 0
    total = len(cluster_df)
    total_skipped = 0

    # Track per-cluster accuracy
    cluster_stats = {c: {"correct": 0, "total": 0} for c in cluster_df["cluster"].unique()}

    print(f"Evaluating on {total} samples...")

    progress = tqdm(cluster_df.itertuples(), total=total, desc="Evaluating", ncols=120)
    tracker = 0
    for row in progress:
        question = row.utterance
        answer = str(row.targetValue).strip()
        cluster = row.cluster
        csv_path = os.path.join(wtq_root, *row.context.split("/"))
        csv_path = os.path.normpath(csv_path)

        # Read table
        try:
            table = pd.read_csv(csv_path, sep=",", quotechar='"', on_bad_lines='skip')
        except Exception as e:
            total_skipped += 1
            print(f"Failed to read CSV: {csv_path}")
            print(f"Error: {e}")
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue

        # Tokenize and generate
        table = table.fillna("").astype(str)
        try:
            inputs = tokenizer(
                table,
                question,
                return_tensors="pt",
                truncation=True,
                padding=True,
            ).to(device)

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=50)

            pred_answer_raw = tokenizer.decode(outputs[0], skip_special_tokens=True)
            pred_answers = [normalize_answer(a) for a in pred_answer_raw.replace(';', ',').split(',') if a.strip()]
            gold_answers = {normalize_answer(answer)}

            if any(a in gold_answers for a in pred_answers):
                correct += 1
                cluster_stats[cluster]["correct"] += 1
            cluster_stats[cluster]["total"] += 1

        except Exception as e:
            print(f"Error processing sample ID {row.id}: {e}")
            total_skipped += 1
            progress.set_postfix({
                "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
                "skipped": total_skipped
            })
            continue

        progress.set_postfix({
            "accuracy": f"{(correct / max(1, progress.n - total_skipped)) * 100:.2f}%",
            "skipped": total_skipped
        })
        tracker += 1

        """ if tracker > 300:
            break """

    global_acc = (correct / max(1, total - total_skipped)) * 100

    # Per-cluster accuracy
    print("\nPer-cluster accuracy:")
    for c, stats in cluster_stats.items():
        if stats["total"] > 0:
            acc = (stats["correct"] / stats["total"]) * 100
            print(f"{c:<15}: {acc:.2f}% ({stats['correct']}/{stats['total']})")

    return global_acc, cluster_stats, total_skipped

In [ ]:
new_train_cluster = "clustered_training.csv"
global_acc, cluster_stats, total_skipped = new_clustered_tapex(model, tokenizer, new_train_cluster, wtq_root, device)